In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"Hidden size: {model.config.hidden_size}")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
dataset = dataset.select(range(len(dataset) - 250, len(dataset)))

print("Dataset split: glue/mrpc validation (last 250 examples)")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def encode_texts(texts, batch_size=64, max_length=128):
    all_embeddings = []
    for start_idx in range(0, len(texts), batch_size):
        batch_texts = texts[start_idx:start_idx + batch_size]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            embeddings = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
            embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

sentence1_list = dataset["sentence1"]
sentence2_list = dataset["sentence2"]
labels = dataset["label"]

emb1 = encode_texts(sentence1_list, batch_size=64, max_length=128)
emb2 = encode_texts(sentence2_list, batch_size=64, max_length=128)

similarity_scores = torch.sum(emb1 * emb2, dim=1).tolist()

threshold = 0.80
predictions = [1 if score >= threshold else 0 for score in similarity_scores]
margins = [score - threshold for score in similarity_scores]
absolute_margins = [abs(m) for m in margins]

print(f"Completed embedding inference for {len(predictions)} examples.")
print("Scoring method: normalized dot product via rowwise matrix multiplication")
print(f"Fixed threshold: {threshold}")

In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

positive_scores = [score for score, label in zip(similarity_scores, labels) if label == 1]
negative_scores = [score for score, label in zip(similarity_scores, labels) if label == 0]
mean_positive_similarity = sum(positive_scores) / len(positive_scores)
mean_negative_similarity = sum(negative_scores) / len(negative_scores)

positive_avg_margin = sum(abs(score - threshold) for score, label in zip(similarity_scores, labels) if label == 1) / sum(1 for label in labels if label == 1)
negative_avg_margin = sum(abs(score - threshold) for score, label in zip(similarity_scores, labels) if label == 0) / sum(1 for label in labels if label == 0)

errors = []
for i, (label, pred, score, margin) in enumerate(zip(labels, predictions, similarity_scores, absolute_margins)):
    if label != pred:
        errors.append({
            "local_index": i,
            "true": label,
            "pred": pred,
            "score": score,
            "abs_margin": margin
        })

errors = sorted(errors, key=lambda x: x["abs_margin"])

print("Evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)
print(f"Mean similarity | label=1: {mean_positive_similarity:.4f}")
print(f"Mean similarity | label=0: {mean_negative_similarity:.4f}")
print(f"Average |score-threshold| | label=1: {positive_avg_margin:.4f}")
print(f"Average |score-threshold| | label=0: {negative_avg_margin:.4f}")
print(f"Total errors: {len(errors)}")

In [ ]:
print("Compact error table (up to 12 closest mistakes to threshold)")
print("local_index\ttrue\tpred\tscore\tabs_margin")
for row in errors[:12]:
    print(f"{row['local_index']}\t{row['true']}\t{row['pred']}\t{row['score']:.4f}\t{row['abs_margin']:.4f}")

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("inference_method=separate_sentence_embeddings_with_normalized_rowwise_dot_product")
print("dataset_split=glue/mrpc validation last_250")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"threshold={threshold}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"mean_positive_similarity={mean_positive_similarity:.4f}")
print(f"mean_negative_similarity={mean_negative_similarity:.4f}")
print(f"avg_margin_label_1={positive_avg_margin:.4f}")
print(f"avg_margin_label_0={negative_avg_margin:.4f}")
print(f"num_errors={len(errors)}")